# Download and convert to Zarr
This downloads SWOT Pixel Cloud products from hydroweb.next (API-Key necessary) based on a region and a period of interest.
Then is extracts information contained in the area of interest for your study, stores everything in a Zarr Database (based on the zcollection package) for future use.
Zarr (and the way we partitionned data with zcollection) is very efficient for computation. However, it is not (yet) compatible with QGIS compared to Geopackage.


## Setting the region and period of interest
Using a geopackage layer, preliminary created with, e.g. QGIS, to limit data download and database

In [10]:
import pixcdust
from pixcdust.downloaders.hydroweb_next import PixCDownloader
import geopandas as gpd
from datetime import datetime

In [11]:
# reading the area of interest polygon
gdf_geom = gpd.read_file("../data/aoi.gpkg")

dates = (
    datetime(2023, 4, 6),
    datetime(2023, 4, 8),
)

## Download
This will unfortunately lead to downloading many big files (that will be removed later). This is the only way right now, but the hydroweb.next team is working on improving that.

In [3]:
pixcdownloader = PixCDownloader(
    gdf_geom,
    dates,
    verbose=1,
    path_download="/tmp/pixc",
)
pixcdownloader.search_download()

Downloaded products:   0%|                                                                                    …

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

## Extraction
Now we have all necessary files, let us extract key variables within area of interest in a Zarr (zcollection) database.
This Zarr partionned format is very efficient for time analysis, but is not currently accessible in GIS softwares such as QGIS
We are using the same geodataframe to limit the data to the area of interest

In [ ]:
from pixcdust.converters.zarr import Nc2ZarrConverter
from glob import glob

In [ ]:
# You can specify conditions on variables to filter data
conditions = {
    "sig0": {"operator": "gt", "threshold": 20},  # sig0 > 20
    "classification": {"operator": "ge", "threshold": 3},  # classification >= 3
}

pixc = Nc2ZarrConverter(
    path_in=glob(pixcdownloader.path_download + "/**/*.nc", recursive=True),
    variables=["height", "sig0", "classification"],
    area_of_interest=gdf_geom,
    conditions=conditions,
)
pixc.database_from_nc(path_out="/tmp/pixc_zarr")

database has been succesfully created, we can remove the raw files

In [7]:
# import shutil
# shutil.rmtree('/tmp/pixc')

# Read the database
previous steps are not necessary

Now we can open this database in a xarray, or dataframe, or GeoDataFrame

In [18]:
from pixcdust.readers.zarr import ZarrReader
import datetime

pixc_read = ZarrReader("/tmp/pixc_zarr")
pixc_read.read((datetime.datetime(2023, 4, 6), datetime.datetime(2023, 4, 8)))
pixc_read.data

/work/scratch/env/simeonma/dev/hawk_custom/.pixi/envs/default/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46081 instead
  warnings.warn(


<xarray.Dataset> Size: 3MB
Dimensions:         (points: 72980)
Dimensions without coordinates: points
Data variables:
    tile_number     (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    pass_number     (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    classification  (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    height          (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    sig0            (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    longitude       (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    latitude        (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    cycle_number    (points) float32 292kB dask.array<chunksize=(34624,), meta=np.ndarray>
    time            (points) datetime64[ns] 584kB dask.array<chunksize=(34624,), meta=np.ndarray>
Attributes:
    azimuth_offset:              3
    description:                 cloud of geolocated interferogram pixels
    interferogram_size_azimuth:  3245
    interferogram_size_range:    4857
    looks_to_efflooks:           1.5340684990936673
    num_azimuth_looks:           7.0

In [19]:
gdf_pixc = pixc_read.to_geodataframe()
gdf_pixc

/work/scratch/env/simeonma/dev/swot_pixc_study/pixcdust/readers/base_reader.py:142: UserWarning: No active geometry column to be set. The resulting object will be a pandas.DataFrame with geopandas.GeometryArray(s) containing geometry and CRS information. Use `.set_geometry()` to set an active geometry and upcast to the geopandas.GeoDataFrame manually.
  gdf = self.data.xvec.to_geodataframe()


,tile_number,pass_number,classification,height,sig0,longitude,latitude,cycle_number,time
points,,,,,,,,,
0,78.0,16.0,3.0,277.479858,20.353701,1.479326,43.523891,482.0,2023-04-06 09:46:18
1,78.0,16.0,3.0,270.444000,28.767691,1.477789,43.524021,482.0,2023-04-06 09:46:18
2,78.0,16.0,6.0,242.566086,47.593857,1.468200,43.531647,482.0,2023-04-06 09:46:18
3,78.0,16.0,6.0,253.108810,45.258018,1.470237,43.532013,482.0,2023-04-06 09:46:18
4,78.0,16.0,3.0,254.432236,47.007401,1.470340,43.532028,482.0,2023-04-06 09:46:18
...,...,...,...,...,...,...,...,...,...
72975,78.0,16.0,3.0,255.025650,36.264046,1.501647,43.675690,483.0,2023-04-07 09:36:56
72976,78.0,16.0,3.0,254.909988,40.682728,1.501748,43.675705,483.0,2023-04-07 09:36:56
72977,78.0,16.0,3.0,263.372375,23.598780,1.504855,43.686821,483.0,2023-04-07 09:36:56


Enjoy!